# Tutorial 1: Basic Genotyping Workflow

This tutorial demonstrates the complete basic workflow for COI sequence genotyping using BOLDGenotyper.

## Learning Objectives

By the end of this tutorial, you will:
- Understand how to run a basic genotyping analysis
- Learn to interpret genotype assignments
- Explore geographic distributions of genotypes
- Navigate the HTML interactive report
- Understand the output file structure

## Prerequisites

- BOLDGenotyper installed (see main README.md)
- Example dataset: `Sphyrna_lewini_scallopedhammerhead.tsv`
- Basic understanding of COI barcoding

## Dataset

We'll use the scalloped hammerhead shark (*Sphyrna lewini*) dataset:
- ~600 COI sequences
- Global distribution
- Species-level analysis
- Runtime: ~3-5 minutes

## Step 1: Verify Installation

First, let's verify that BOLDGenotyper is properly installed.

In [ ]:
# Check BOLDGenotyper version
!boldgenotyper --version

In [ ]:
# Verify all dependencies
import sys
import pandas as pd
import numpy as np
from pathlib import Path

print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# Check for external dependencies
!vsearch --version 2>&1 | head -1
!mafft --version 2>&1 | head -1

## Step 2: Examine Input Data

Let's look at the structure of the input data from BOLD.

In [ ]:
# Load the input TSV file
input_file = "../data/Sphyrna_lewini_scallopedhammerhead.tsv"
df = pd.read_csv(input_file, sep='\t', low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check key columns for genotyping
print("Key columns for genotyping:")
print(f"- Process IDs: {df['processid'].nunique()} unique")
print(f"- Species: {df['species_name'].nunique()} unique")
print(f"- Sequences: {df['nucleotides'].notna().sum()} with sequence data")
print(f"- Coordinates: {df['lat'].notna().sum()} with latitude")

# Check geographic distribution
print(f"\nGeographic distribution:")
print(df['country'].value_counts().head(10))

## Step 3: Run Basic Genotyping Analysis

Now we'll run the main BOLDGenotyper pipeline with default parameters.

### Default Parameters:
- Clustering threshold: 0.03 (3% divergence)
- Similarity threshold: 0.97 (97% identity for assignment)
- Tie margin: 0.001 (0.1% for tie detection)
- Geographic assignment: Ocean basins (GOaS)

In [ ]:
# Run BOLDGenotyper with default parameters
!boldgenotyper ../data/Sphyrna_lewini_scallopedhammerhead.tsv \
  --output ../data/Sphyrna_lewini_tutorial/ \
  --threads 4

## Step 4: Explore Output Files

Let's examine the output directory structure.

In [ ]:
# List output directory structure
!tree ../data/Sphyrna_lewini_tutorial/ -L 2

In [ ]:
# Load the main output file: annotated genotype assignments
output_file = "../data/Sphyrna_lewini_tutorial/Sphyrna_lewini_annotated.csv"
results = pd.read_csv(output_file)

print(f"Results shape: {results.shape}")
print(f"\nColumns: {results.columns.tolist()}")
print(f"\nFirst few rows:")
results.head()

## Step 5: Interpret Genotype Assignments

Let's analyze the genotype distribution and assignment quality.

In [ ]:
# Count genotypes
print(f"Total genotypes identified: {results['genotype'].nunique()}")
print(f"Total samples assigned: {len(results)}")
print(f"\nGenotype frequency distribution:")
genotype_counts = results['genotype'].value_counts()
print(genotype_counts.head(10))

In [ ]:
# Visualize genotype sizes
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of genotype sizes
axes[0].hist(genotype_counts.values, bins=30, edgecolor='black')
axes[0].set_xlabel('Samples per Genotype')
axes[0].set_ylabel('Number of Genotypes')
axes[0].set_title('Genotype Size Distribution')
axes[0].set_yscale('log')

# Top 20 genotypes
genotype_counts.head(20).plot(kind='bar', ax=axes[1])
axes[1].set_xlabel('Genotype')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Top 20 Most Abundant Genotypes')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Check assignment quality
print("Assignment quality metrics:")
print(f"\nAssignment status:")
print(results['assignment_status'].value_counts())

if 'percent_identity' in results.columns:
    print(f"\nPercent identity statistics:")
    print(results['percent_identity'].describe())

# Check for ties (ambiguous assignments)
if 'tie_flag' in results.columns:
    n_ties = results['tie_flag'].sum()
    print(f"\nAmbiguous assignments (ties): {n_ties} ({n_ties/len(results)*100:.1f}%)")

## Step 6: Analyze Geographic Distribution

Explore how genotypes are distributed across ocean basins.

In [ ]:
# Check ocean basin assignments
if 'ocean_basin' in results.columns:
    print("Ocean basin distribution:")
    print(results['ocean_basin'].value_counts())
    
    # Cross-tabulation of genotypes by ocean basin
    print("\nGenotypes per ocean basin:")
    geo_genotype = pd.crosstab(results['ocean_basin'], results['genotype'])
    print(f"Shape: {geo_genotype.shape}")
    print(f"\nTop 5 genotypes by basin:")
    for basin in geo_genotype.index[:5]:
        top_genotypes = geo_genotype.loc[basin].nlargest(3)
        print(f"\n{basin}:")
        print(top_genotypes)

In [ ]:
# Visualize geographic distribution
if 'ocean_basin' in results.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Count samples per basin
    basin_counts = results['ocean_basin'].value_counts()
    basin_counts.plot(kind='barh', ax=ax)
    ax.set_xlabel('Number of Samples')
    ax.set_ylabel('Ocean Basin')
    ax.set_title('Sample Distribution Across Ocean Basins')
    
    plt.tight_layout()
    plt.show()

## Step 7: Examine Diagnostics

Load and review the diagnostics file for assignment quality.

In [ ]:
# Load diagnostics file
diagnostics_file = "../data/Sphyrna_lewini_tutorial/genotype_assignments/Sphyrna_lewini_diagnostics.csv"
diagnostics = pd.read_csv(diagnostics_file)

print(f"Diagnostics shape: {diagnostics.shape}")
print(f"\nColumns: {diagnostics.columns.tolist()}")
print(f"\nFirst few rows:")
diagnostics.head()

In [ ]:
# Analyze identity distribution
if 'best_identity' in diagnostics.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Best identity distribution
    axes[0].hist(diagnostics['best_identity'], bins=50, edgecolor='black')
    axes[0].axvline(0.97, color='red', linestyle='--', label='Similarity threshold (97%)')
    axes[0].set_xlabel('Best Identity (%)')
    axes[0].set_ylabel('Number of Samples')
    axes[0].set_title('Assignment Identity Distribution')
    axes[0].legend()
    
    # Delta identity (difference between best and second-best)
    if 'delta_identity' in diagnostics.columns:
        axes[1].hist(diagnostics['delta_identity'], bins=50, edgecolor='black')
        axes[1].axvline(0.001, color='red', linestyle='--', label='Tie margin (0.1%)')
        axes[1].set_xlabel('Delta Identity (Best - Second Best)')
        axes[1].set_ylabel('Number of Samples')
        axes[1].set_title('Assignment Confidence')
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## Step 8: View Interactive HTML Report

BOLDGenotyper generates an interactive HTML report. Open it in your browser.

In [ ]:
# Get path to HTML report
html_report = Path("../data/Sphyrna_lewini_tutorial/Sphyrna_lewini_report.html")

if html_report.exists():
    print(f"Interactive HTML report available at:")
    print(f"{html_report.absolute()}")
    print(f"\nOpen this file in your web browser to explore:")
    print("- Interactive maps")
    print("- Genotype distribution plots")
    print("- Quality control metrics")
    print("- Sample metadata tables")
    
    # Display in Jupyter (if using JupyterLab)
    from IPython.display import IFrame
    display(IFrame(src=str(html_report), width=1000, height=600))
else:
    print("HTML report not found. Check pipeline output.")

## Step 9: Summary Statistics

Generate a summary of the analysis.

In [ ]:
# Create summary report
print("="*60)
print("BOLDGenotyper Analysis Summary")
print("="*60)
print(f"\nDataset: Sphyrna lewini (scalloped hammerhead shark)")
print(f"Input samples: {len(df)}")
print(f"Samples with sequences: {df['nucleotides'].notna().sum()}")
print(f"\nGenotyping Results:")
print(f"  Total genotypes: {results['genotype'].nunique()}")
print(f"  Total assigned samples: {len(results)}")
print(f"  Largest genotype: {genotype_counts.max()} samples")
print(f"  Singleton genotypes: {(genotype_counts == 1).sum()}")

if 'ocean_basin' in results.columns:
    print(f"\nGeographic Distribution:")
    print(f"  Ocean basins represented: {results['ocean_basin'].nunique()}")
    print(f"  Most sampled basin: {results['ocean_basin'].value_counts().index[0]}")

if 'assignment_status' in results.columns:
    assigned = (results['assignment_status'] == 'assigned').sum()
    print(f"\nAssignment Quality:")
    print(f"  Successfully assigned: {assigned} ({assigned/len(results)*100:.1f}%)")

print(f"\nOutput files in: ../data/Sphyrna_lewini_tutorial/")
print("="*60)

## Key Takeaways

1. **Genotype Assignment**: BOLDGenotyper identified genotypes using sequence clustering at 3% divergence
2. **Quality Control**: Most samples were assigned with high confidence (>97% identity)
3. **Geographic Patterns**: Genotypes show ocean basin-specific distributions
4. **Output Files**: Multiple formats available for downstream analysis

## Next Steps

- **Tutorial 2**: Learn how to optimize the clustering threshold using parameter sweep
- **Tutorial 3**: Perform quality control using comparative analysis
- **Tutorial 4**: Use custom shapefiles for non-marine organisms
- **Tutorial 5**: Export data for population genetics analysis

## Additional Resources

- Main README: `../README.md`
- Parameter reference: See README sections 6.3-6.4
- Output file guide: See README section 7
- API reference: `../API_REFERENCE.md`